In [7]:
# Cell 1: get the code and the training stack
!git clone https://github.com/SachiK0thari/care-nudge.git
%cd care-nudge
!pip install -q -r requirements-dev.txt

# Cell 1: fresh copy of the code and the training stack
%cd /content
!rm -rf /content/care-nudge
!git clone https://github.com/SachiK0thari/care-nudge.git
%cd /content/care-nudge
!pip install -q -r requirements-dev.txt


fatal: destination path 'care-nudge' already exists and is not an empty directory.
/content/care-nudge


In [8]:
# Cell 2: log in to Hugging Face (paste a write token)
from huggingface_hub import login
login()


In [9]:
!nvidia-smi
import torch; print(torch.cuda.is_available())

Tue Nov 17 06:03:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Cell 3: data + two baselines from the un-tuned base model: the same prompt, then the prompt plus worked examples.
!python scripts/build_dataset.py
# The two baselines take about 30 min each on a T4. Comment them out if reports/eval_base*.json already exist.
!python scripts/evaluate.py --model Qwen/Qwen3-4B-Instruct-2507 --name base
!python scripts/evaluate.py --model Qwen/Qwen3-4B-Instruct-2507 --few-shot --name base_fewshot

val       63 -> data/val.jsonl
train    570 -> data/train.jsonl
test     260 -> data/test.jsonl
probes    12 -> data/probes.jsonl
train rows per level: ROUTINE 96, SOON 132, URGENT 130, EMERGENCY 212
distinct replies: 30 across 570 rows (most repeated: 24x)
Loading weights: 100% 398/398 [00:00<00:00, 16580.58it/s]


In [15]:
# Cell 4: fine-tune, then evaluate the adapter on the test split and on the probe set
# && stops the chain if a step fails, so evaluate/merge never run against a missing adapter
!python scripts/train.py --push-to SachiK/carenudge-lora && \
  python scripts/evaluate.py --model Qwen/Qwen3-4B-Instruct-2507 --adapter outputs/carenudge-lora --name finetuned && \
  python scripts/evaluate.py --model Qwen/Qwen3-4B-Instruct-2507 --adapter outputs/carenudge-lora --test data/probes.jsonl --name finetuned_probes

Loading weights: 100% 398/398 [00:34<00:00, 11.69it/s]
  0% 0/108 [00:00<?, ?it/s]Traceback (most recent call last):
  File "/content/care-nudge/care-nudge/care-nudge/care-nudge/scripts/train.py", line 89, in <module>
    main()
    ~~~~^^
  File "/content/care-nudge/care-nudge/care-nudge/care-nudge/scripts/train.py", line 78, in main
    trainer.train()
    ~~~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/transformers/trainer.py", line 1559, in train
    return inner_training_loop(
        args=args,
    ...<2 lines>...
        ignore_keys_for_eval=ignore_keys_for_eval,
    )
  File "/usr/local/lib/python3.13/dist-packages/transformers/trainer.py", line 1643, in _inner_training_loop
    self._run_epoch(
    ~~~~~~~~~~~~~~~^
        model=model,
        ^^^^^^^^^^^^
    ...<9 lines>...
        steps_trained_in_current_epoch=steps_trained_in_current_epoch,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/

In [ ]:
!python scripts/train.py 2>&1 | tail -n 60

In [ ]:
# Cell 5: merge the adapter and publish the model the Space will load
![ -f outputs/carenudge-lora/adapter_config.json ] && python scripts/merge.py --push-to SachiK/carenudge-qwen3-4b || echo "No adapter in outputs/carenudge-lora: Cell 4 training did not finish, skipping merge."